<a href="https://colab.research.google.com/github/Rollybuilds/employee-payroll-system/blob/main/Employee_Payroll_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

** PROJECT 4 - EMPLOYEE PAYROLL SYSTEM**

In [1]:
import csv
from abc import ABC, abstractmethod
from datetime import datetime

EMPLOYEES_FILE = "employees.csv"
PAYSLIPS_FILE = "payslips.txt"


# ---------------------------------------------------------------- Exception
class PayrollError(Exception):
    """Raised when a payroll action cannot be completed."""


# ------------------------------------------------- Abstract base class (abc)
class Employee(ABC):
    """Base class for all employees."""

    def __init__(self, emp_id, name, base_pay):
        self.__emp_id = emp_id        # private attributes (encapsulation)
        self.__name = ""
        self.__base_pay = 0
        self.name = name              # uses the setters below for validation
        self.base_pay = base_pay

    # Getters and setters
    @property
    def emp_id(self):
        return self.__emp_id

    @property
    def name(self):
        return self.__name

    @name.setter
    def name(self, value):
        if not value.strip():
            raise ValueError("Employee name cannot be empty.")
        self.__name = value.strip()

    @property
    def base_pay(self):
        return self.__base_pay

    @base_pay.setter
    def base_pay(self, value):
        if value <= 0:
            raise ValueError("Pay must be greater than zero.")
        self.__base_pay = value

    # Salary calculation (same for every employee type)
    def gross_salary(self):
        return sum(self.earnings().values())

    def total_deductions(self):
        return sum(self.deductions().values())

    def net_salary(self):
        return self.gross_salary() - self.total_deductions()

    def to_row(self):
        return [self.emp_id, self.name, self.employee_type(),
                f"{self.base_pay:.2f}", 0]

    def __str__(self):
        return f"[{self.emp_id}] {self.name} | {self.employee_type()} | Pay: {self.base_pay:.2f}"

    # Every child class must implement these
    @abstractmethod
    def employee_type(self):
        """Return the type of the employee."""

    @abstractmethod
    def earnings(self):
        """Return a dictionary of earnings, e.g. {'Basic': 20000}."""

    @abstractmethod
    def deductions(self):
        """Return a dictionary of deductions, e.g. {'PF': 2400}."""


# ------------------------------------------------ Derived classes (inheritance)
class FullTimeEmployee(Employee):
    """Monthly salary. base_pay is the basic monthly salary."""

    def employee_type(self):
        return "Full-Time"

    def earnings(self):
        return {
            "Basic Salary": self.base_pay,
            "HRA (20%)": self.base_pay * 0.20,
            "DA (10%)": self.base_pay * 0.10,
        }

    def deductions(self):
        return {"PF (12%)": self.base_pay * 0.12}


class PartTimeEmployee(Employee):
    """Hourly pay. base_pay is the pay per hour."""

    def __init__(self, emp_id, name, base_pay, hours_worked=0):
        super().__init__(emp_id, name, base_pay)
        self.__hours_worked = 0
        self.hours_worked = hours_worked

    @property
    def hours_worked(self):
        return self.__hours_worked

    @hours_worked.setter
    def hours_worked(self, hours):
        if hours < 0:
            raise ValueError("Hours worked cannot be negative.")
        self.__hours_worked = hours

    def employee_type(self):
        return "Part-Time"

    def earnings(self):
        return {f"Pay ({self.hours_worked:g} hrs x {self.base_pay:g})":
                self.base_pay * self.hours_worked}

    def deductions(self):
        return {}

    def to_row(self):
        return [self.emp_id, self.name, self.employee_type(),
                f"{self.base_pay:.2f}", self.hours_worked]


# ---------------------------------------------------------------- Payroll
class Payroll:
    """Manages all employees and saves them to a CSV file."""

    def __init__(self):
        self.__employees = {}
        self.load_data()

    # ----- file handling
    def load_data(self):
        try:
            with open(EMPLOYEES_FILE, newline="") as file:
                for row in csv.DictReader(file):
                    emp_id, name = int(row["emp_id"]), row["name"]
                    pay = float(row["base_pay"])
                    if row["type"] == "Part-Time":
                        employee = PartTimeEmployee(emp_id, name, pay,
                                                    float(row["hours"]))
                    else:
                        employee = FullTimeEmployee(emp_id, name, pay)
                    self.__employees[emp_id] = employee
        except FileNotFoundError:
            print("No saved employees found. Starting with an empty list.")
        except (KeyError, ValueError):
            print("Some employee records could not be read.")

    def save_data(self):
        try:
            with open(EMPLOYEES_FILE, "w", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(["emp_id", "name", "type", "base_pay", "hours"])
                for employee in self.__employees.values():
                    writer.writerow(employee.to_row())
        except OSError:
            print("Error: could not save employee data.")

    # ----- payroll actions
    def add_employee(self, name, emp_type, pay):
        new_id = max(self.__employees, default=100) + 1
        if emp_type == "Full-Time":
            employee = FullTimeEmployee(new_id, name, pay)
        else:
            employee = PartTimeEmployee(new_id, name, pay)
        self.__employees[new_id] = employee
        self.save_data()
        return employee

    def find_employee(self, emp_id):
        if emp_id not in self.__employees:
            raise PayrollError("No employee found with that ID.")
        return self.__employees[emp_id]

    def search(self, keyword):
        keyword = keyword.strip().lower()
        return [e for e in self.__employees.values()
                if keyword in e.name.lower() or keyword == str(e.emp_id)]

    def all_employees(self):
        return list(self.__employees.values())

    def generate_payslip(self, employee):
        """Build the payslip text and save it to the payslips file."""
        line = "=" * 36
        lines = [
            line,
            "PAYSLIP".center(36),
            line,
            f"Employee ID : {employee.emp_id}",
            f"Name        : {employee.name}",
            f"Type        : {employee.employee_type()}",
            f"Month       : {datetime.now().strftime('%B %Y')}",
            "-" * 36,
            "EARNINGS",
        ]
        for label, amount in employee.earnings().items():
            lines.append(f"  {label:<24}{amount:>10.2f}")
        lines.append(f"  {'Gross Salary':<24}{employee.gross_salary():>10.2f}")
        lines.append("DEDUCTIONS")
        for label, amount in employee.deductions().items():
            lines.append(f"  {label:<24}{amount:>10.2f}")
        lines.append(f"  {'Total Deductions':<24}{employee.total_deductions():>10.2f}")
        lines.append("-" * 36)
        lines.append(f"{'NET SALARY':<26}{employee.net_salary():>10.2f}")
        lines.append(line)
        payslip = "\n".join(lines)

        try:
            with open(PAYSLIPS_FILE, "a") as file:
                file.write(payslip + "\n\n")
        except OSError:
            print("Error: could not save the payslip to file.")
        return payslip


# ---------------------------------------------------------------- Menu
def read_number(message, number_type=int):
    """Keep asking until the user enters a valid number."""
    while True:
        try:
            return number_type(input(message))
        except ValueError:
            print("Please enter a valid number.")


def ask_hours_if_part_time(payroll, employee):
    """Part-time pay depends on the hours worked, so ask for them first."""
    if employee.employee_type() == "Part-Time":
        employee.hours_worked = read_number("Hours worked this month: ", float)
        payroll.save_data()


def show_employees(employees):
    if not employees:
        print("No employees found.")
    for employee in employees:
        print(employee)


def main():
    payroll = Payroll()

    menu = """
===== EMPLOYEE PAYROLL SYSTEM =====
1. Add employee
2. Calculate salary
3. Generate payslip
4. Search employees
5. Display all employees
6. Exit
"""

    while True:
        print(menu)
        choice = input("Enter your choice (1-6): ").strip()

        try:
            if choice == "1":
                name = input("Employee name: ")
                kind = input("Type (1-Full-Time, 2-Part-Time): ").strip()
                if kind not in ("1", "2"):
                    raise PayrollError("Invalid employee type.")
                if kind == "1":
                    pay = read_number("Basic monthly salary: ", float)
                    employee = payroll.add_employee(name, "Full-Time", pay)
                else:
                    pay = read_number("Pay per hour: ", float)
                    employee = payroll.add_employee(name, "Part-Time", pay)
                print(f"Employee added with ID {employee.emp_id}.")

            elif choice == "2":
                employee = payroll.find_employee(read_number("Employee ID: "))
                ask_hours_if_part_time(payroll, employee)
                print(f"{employee.name} | Gross: {employee.gross_salary():.2f} | "
                      f"Deductions: {employee.total_deductions():.2f} | "
                      f"Net: {employee.net_salary():.2f}")

            elif choice == "3":
                employee = payroll.find_employee(read_number("Employee ID: "))
                ask_hours_if_part_time(payroll, employee)
                print(payroll.generate_payslip(employee))
                print(f"Payslip saved in {PAYSLIPS_FILE}.")

            elif choice == "4":
                show_employees(payroll.search(input("Search by name or ID: ")))

            elif choice == "5":
                show_employees(payroll.all_employees())

            elif choice == "6":
                payroll.save_data()
                print("Data saved. Goodbye!")
                break

            else:
                print("Invalid choice. Please select 1-6.")

        except (PayrollError, ValueError) as error:
            print(f"Error: {error}")


if __name__ == "__main__":
    main()

No saved employees found. Starting with an empty list.

===== EMPLOYEE PAYROLL SYSTEM =====
1. Add employee
2. Calculate salary
3. Generate payslip
4. Search employees
5. Display all employees
6. Exit

Enter your choice (1-6): 1
Employee name: Umang
Type (1-Full-Time, 2-Part-Time): 1
Basic monthly salary: 25000
Employee added with ID 101.

===== EMPLOYEE PAYROLL SYSTEM =====
1. Add employee
2. Calculate salary
3. Generate payslip
4. Search employees
5. Display all employees
6. Exit

Enter your choice (1-6): 1
Employee name: Kajal
Type (1-Full-Time, 2-Part-Time): 1
Basic monthly salary: 30000
Employee added with ID 102.

===== EMPLOYEE PAYROLL SYSTEM =====
1. Add employee
2. Calculate salary
3. Generate payslip
4. Search employees
5. Display all employees
6. Exit

Enter your choice (1-6): 2
Employee ID: 101
Umang | Gross: 32500.00 | Deductions: 3000.00 | Net: 29500.00

===== EMPLOYEE PAYROLL SYSTEM =====
1. Add employee
2. Calculate salary
3. Generate payslip
4. Search employees
5. Disp